In [1]:
import pickle 

In [4]:
with open('../data/corpus_all.pkl','rb') as f:
    corpus=pickle.load(f)

In [28]:
corpus[379]

'The scene is laid in the house of Cephalus at the Piraeus; and the\nwhole dialogue is narrated by Socrates the day after it actually took\nplace to Timaeus, Hermocrates, Critias, and a nameless person, who are\nintroduced in the Timaeus.'

In [29]:
with open('../data/vector_space.pkl','rb') as f:
    vector_space=pickle.load(f)

In [30]:
vector_space

array([[-0.02754853,  0.03830466,  0.02392452, ...,  0.02693123,
         0.07863498, -0.00170038],
       [-0.00739761,  0.00069855, -0.05868614, ...,  0.03592109,
         0.00788429, -0.03979349],
       [-0.10558078,  0.08086269, -0.00465923, ...,  0.02838872,
        -0.00192587, -0.02106022],
       ...,
       [-0.01604316,  0.05398728, -0.06858031, ..., -0.00706531,
         0.02678093, -0.04382266],
       [-0.03381354,  0.0333888 , -0.06110155, ...,  0.10103336,
         0.01593934, -0.0524285 ],
       [-0.06717259,  0.13432932, -0.06080251, ...,  0.15992054,
         0.04044572, -0.07554927]], shape=(377, 384))

In [1]:
import pickle
import logging
from typing import Dict, List
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams,
    PointStruct, SparseVector
)

In [2]:
client = QdrantClient("http://localhost:6333")

In [3]:
dense_model = SentenceTransformer("all-MiniLM-L6-v2")

In [4]:
collection_name='simple_rag_hybrid'
test_query = "who is socrates?"

    # Test dense search
dense_results = client.query_points(
        collection_name=collection_name,
        query=dense_model.encode(test_query).tolist(),
        limit=3,
        using='dense',
        with_payload=True
    )
print(f"Dense search results: {len(dense_results.points)}")



Dense search results: 3


In [9]:
%cd /home/hagusta/workspace/simple_rag

/home/hagusta/workspace/simple_rag


In [11]:
from sparse_utils import BM25Encoder

In [15]:
sparse_encoder=BM25Encoder()

In [28]:
with open('data/corpus_all.pkl', 'rb') as f:
        corpus = pickle.load(f)

corpus_texts = list(corpus.values())
sparse_encoder.fit(corpus_texts)

In [30]:
sparse_vector = sparse_encoder.encode(test_query)
sparse_results = client.query_points(
            collection_name=collection_name,
            query=SparseVector(
                indices=list(sparse_vector.keys()),
                values=list(sparse_vector.values())
            ),
            limit=3,
            using='sparse',
            with_payload=True
        )

print(f"Sparse search results: {len(sparse_results.points)}")


Sparse search results: 3


In [31]:
sparse_results

QueryResponse(points=[ScoredPoint(id=619, version=0, score=1325.6333, payload={'text': '"The god of the Old Testament is arguably the most unpleasant character in all fiction:..." Well well well well well hmmmmmm considering the absolute fact that you cannot prove that YOUR god even exists, YOUR entire debate is entirely irrelevant on those grounds alone, you"ve automatically lost the debate. Oh and Btw, I didn"t make the opening statement for RD1. Richard Dawkins did. He happens to know a lot more about YOUR god, religion and bible than you ---ever--- will. As the matter of fact, so do I. Now let"s see where your debate leads. So let"s get started. - This is not a debate about whether God exists or not, there is no need to make this discussion personal, how can I loose a debate that is about if God has mercy or not in the old testament because you claim that he does not exist? Remember, I am only here to debate and hopefully win, it does not matter to me what your views on God are, Al